# Notebook 2: Domain Separation

**Pipeline step:** Extract cupin and MetRS domain sequences from the full-length HS proteins and write them to FASTA files.

**Overview:** Hydrazine synthetase enzymes are multi-domain proteins. Before predicting structures or comparing active sites, we must separate those domains. This notebook explains what protein domains are, how the HS domain boundaries are detected, and how `separate_domains.py` was used to produce the FASTA files for Boltz2 input.

---

## Learning objectives

- Understand what protein domains are and why separating them matters
- Know the specific architecture of HS enzymes (cupin barrel and MetRS fold)
- Read and write FASTA files in Python
- Understand domain annotation from NCBI CDD and HMMER/Pfam
- Walk through the key functions of `separate_domains.py`
- Examine the output FASTA files and verify their contents

---

## 1. What are protein domains?

A protein domain is an independently folding, structurally and functionally distinct unit within a larger protein. Think of a protein as a Swiss army knife: the main body is the full-length chain, but the individual tools — the blade, the screwdriver, the scissors — are the domains. Each tool has its own shape and function, and can even exist as a standalone protein in other organisms.

Domains are important for our analysis because:
- Structure prediction works best on individual domains (shorter sequences fold more reliably)
- Active site comparisons across enzymes are only meaningful within the same domain
- The two HS domains have completely different evolutionary origins and functions

> **Key concept:** The cupin and MetRS domains of HS enzymes have different folds, different metal cofactors, and different substrates. Analysing them separately is not just convenient — it is biologically necessary.

---

## 2. Architecture of hydrazine synthetase

Hydrazine synthetase (HS) is a bifunctional enzyme found in bacteria that produce hydrazine-containing natural products. Its architecture is:

```
N-terminus                                          C-terminus
┌─────────────────┬─────────────────────────────────────────┐
│  Cupin domain   │          MetRS domain                   │
│  (~100-130 aa)  │          (~400-550 aa)                   │
│  β-barrel fold  │  Class I aminoacyl-tRNA synthetase fold  │
│  1 Zn²⁺        │  2 Zn²⁺                                  │
└─────────────────┴─────────────────────────────────────────┘
  e.g. aa 28-101       e.g. aa 121-646  (PyrN numbering)
```

**Cupin domain** — named after the Latin *cupa* (barrel). Forms an anti-parallel β-barrel that coordinates a single zinc ion. Related to cupin superfamily proteins found across all kingdoms. Thought to be involved in substrate binding.

**MetRS domain** — resembles methionyl-tRNA synthetase (MetG), a Class I aminoacyl-tRNA synthetase. Contains the canonical HVGH and KMSKS motifs, and coordinates two zinc ions. This is the catalytic domain that forms the hydrazine bond.

Not all HS-like proteins are di-domain: some bacteria encode the cupin and MetRS subunits separately (cupin-only or MetRS-only proteins).

---

## 3. FASTA format

FASTA is the universal format for sequence data. Each sequence has:
1. A **header line** starting with `>`, containing the sequence identifier and optional description
2. One or more **sequence lines** (conventionally wrapped at 60 or 80 characters)

```
>Spb40|Streptomyces sp. SoC090715LN-17|28-101
GLGVGAGWGRVAPGASSHPDRHDETEFFVIVAGTGELTVDGRRHPVRSGTVVLFEPFESHI
ITNTGETDLVFLTQYWRDSARAR
>PyrN|Streptomyces candidus|40-108
PAWAEAAGVPFNVRRVTLRPGETTAEHNHHDLEVWVMLDGVGEVGWDGHQRVLTAGDSVYL
PPLAPHTLRNLSSDRPLSFFSMWWENL
```

The header format used in our pipeline is: `>Name|Organism|range`

> **Key concept:** FASTA is the lingua franca of sequence bioinformatics. Every downstream tool — BLAST, HMMER, AlphaFold, Boltz2 — accepts FASTA as input.

---

## 4. Domain annotation: NCBI CDD and HMMER

**NCBI CDD (Conserved Domain Database)** annotates known domain families in GenBank records as `Region` features. When a deposited protein matches a cupin or MetRS profile, NCBI records the start/end residues directly in the GenBank record. `fetch_hs_entry.py` (Step 1) reads those `Region` features and stores the ranges in the CSV.

However, not all GenBank records have CDD annotations. For those entries (where `Cupin domain range` and `MetRS domain range` are both `NaN`), the pipeline falls back to **HMMER**.

**HMMER** is a software package that uses **profile hidden Markov models (pHMMs)** to detect sequence homology. A pHMM is a statistical model trained on many aligned sequences from the same family — here, the cupin and MetRS families from **Pfam** (a database of protein family HMMs).

HMMER's `hmmscan` command scans one sequence against a library of HMMs and reports:
- Which domain families match
- The alignment start and end positions (the domain boundaries)

This is more sensitive than BLAST for divergent sequences — it can detect the MetRS domain even at 25% sequence identity to the reference.

---

## 5. Key functions in `separate_domains.py`

Let us read the script and walk through its core logic.

In [3]:
# ── Read and display the script source ───────────────────────────────
SCRIPT_PATH = "/scratch/p318738/SSA/AAR-COMP-012/AAR-COMP-012-Step2/separate_domains.py"

with open(SCRIPT_PATH) as fh:
    source = fh.read()

# Print just the three key functions (lines 150-240 approximately)
lines = source.split("\n")
for i, line in enumerate(lines[149:238], start=150):
    print("{:4d}  {}".format(i, line))

 150  def extract_domain(sequence, range_str):
 151      """Slice sequence using '28-101' range string (1-based inclusive)."""
 152      start, end = map(int, range_str.split("-"))
 153      return sequence[start - 1:end]
 154  
 155  
 156  def resolve_entry(row):
 157      """
 158      Determine cupin and MetRS domain sequences for one CSV row.
 159      Returns (cupin_seq, metrs_seq, updated_cupin_range, updated_metrs_range, method)
 160      """
 161      seq        = row["Sequence"]
 162      arch       = row["Domain Architecture"]
 163      cupin_rng  = row["Cupin domain range"]
 164      metrs_rng  = row["MetRS domain range"]
 165      name       = row["Hydrazine synthase name"]
 166  
 167      cupin_seq  = None
 168      metrs_seq  = None
 169      method     = "annotated"
 170  
 171      # ── case 1: both ranges present (di-domain with annotations) ──────────────
 172      if cupin_rng != "NaN" and metrs_rng != "NaN":
 173          cupin_seq = extract_domain(seq, cupin_rng)

### `extract_domain(sequence, range_str)`

```python
def extract_domain(sequence, range_str):
    """Slice sequence using '28-101' range string (1-based inclusive)."""
    start, end = map(int, range_str.split("-"))  # "28-101" → 28, 101
    return sequence[start - 1:end]               # convert to 0-based Python slicing
```

**Note on indexing:** GenBank and the CSV use 1-based inclusive coordinates (residue 1 is the first amino acid). Python strings are 0-based and the end of a slice is exclusive. So position 28 in biology = index 27 in Python: `sequence[28-1:101]`.

### `fasta_header(row, domain, range_str)`

```python
def fasta_header(row, domain, range_str):
    """Format: >Name|Organism|range"""
    name     = row["Hydrazine synthase name"]
    organism = row["Organism"]
    return ">{}|{}|{}".format(name, organism, range_str)
```

### `resolve_entry(row)`

This function contains the decision logic — which of the four cases applies:
1. Both cupin and MetRS ranges present → use them directly (annotated)
2. Only cupin range → extract cupin only
3. Only MetRS range → extract MetRS only
4. Neither range → run `hmmscan`, or infer from sequence length as a last resort

In [5]:
# ── Implement extract_domain() ourselves and test it ──────────────────

def extract_domain(sequence, range_str):
    """Slice a protein sequence using a '28-101' range string.
    Uses 1-based inclusive coordinates (same as GenBank/CSV).
    """
    parts = range_str.split("-")      # split "28-101" into ["28", "101"]
    start = int(parts[0])             # convert start to integer
    end   = int(parts[1])             # convert end to integer
    # Python slicing is 0-based, end is exclusive
    # 1-based inclusive [28, 101] → Python [27:101]
    return sequence[start - 1 : end]


# Test with PyrN from the database
import pandas as pd

DB_PATH = "/scratch/p318738/SSA/AAR-COMP-012/AAR-COMP-012-Step1/HS_database.csv"
db = pd.read_csv(DB_PATH)

# Get the PyrN row
pyrn = db[db["Hydrazine synthase name"] == "PyrN"].iloc[0]

# Extract the cupin domain
full_seq    = pyrn["Sequence"]
cupin_range = pyrn["Cupin domain range"]   # "40-108"
metrs_range = pyrn["MetRS domain range"]   # "131-640"

cupin_seq = extract_domain(full_seq, cupin_range)
metrs_seq = extract_domain(full_seq, metrs_range)

print("PyrN cupin domain ({}):".format(cupin_range))
print("  {} aa".format(len(cupin_seq)))
print("  {}".format(cupin_seq[:40] + "..."))
print()
print("PyrN MetRS domain ({}):".format(metrs_range))
print("  {} aa".format(len(metrs_seq)))
print("  {}".format(metrs_seq[:40] + "..."))

FileNotFoundError: [Errno 2] No such file or directory: '/scratch/p318738/SSA/AAR-COMP-012/AAR-COMP-012-Step1/HS_database.csv'

---

## 6. Reading and parsing a FASTA file in Python

The output FASTA files from Step 2 are the inputs to Step 3 (Boltz2 structure prediction). Let us read them with Biopython.

In [ ]:
# ── Parse a FASTA file with Biopython ────────────────────────────────
from Bio import SeqIO

CUPIN_FASTA = "/scratch/p318738/SSA/AAR-COMP-012/AAR-COMP-012-Step2/cupin_domains.fasta"

cupin_records = list(SeqIO.parse(CUPIN_FASTA, "fasta"))

print("Total cupin domain sequences: {}".format(len(cupin_records)))
print()

# Each record has .id (the header after '>') and .seq (the sequence)
print("{:<12} {:<40} {:>8}".format("Name", "Organism (partial)", "Length"))
print("-" * 64)
for rec in cupin_records:
    # Header format: Name|Organism|range
    parts    = rec.id.split("|")
    name     = parts[0]
    organism = parts[1][:38] if len(parts) > 1 else "?"
    length   = len(rec.seq)
    print("{:<12} {:<40} {:>8}".format(name, organism, length))

In [ ]:
# ── Do the same for MetRS domains ─────────────────────────────────────
METRS_FASTA = "/scratch/p318738/SSA/AAR-COMP-012/AAR-COMP-012-Step2/metrs_domains.fasta"

metrs_records = list(SeqIO.parse(METRS_FASTA, "fasta"))

print("Total MetRS domain sequences: {}".format(len(metrs_records)))
print()

# Print length statistics
lengths = [len(r.seq) for r in metrs_records]
print("Length range: {} - {} aa".format(min(lengths), max(lengths)))
print("Mean length:  {:.0f} aa".format(sum(lengths) / len(lengths)))

---

## 7. Writing a FASTA file manually

It is useful to understand how FASTA files are written — `separate_domains.py` does it with a simple loop.

In [ ]:
# ── Write a minimal FASTA file from scratch ───────────────────────────

def write_fasta(path, entries, wrap=60):
    """
    Write a FASTA file.
    entries : list of (header_string, sequence_string)
    wrap    : characters per sequence line
    """
    with open(path, "w") as fh:
        for header, seq in entries:
            fh.write(">" + header + "\n")    # write header line
            # Write sequence in chunks of 'wrap' characters
            for i in range(0, len(seq), wrap):
                fh.write(seq[i:i + wrap] + "\n")


# Build entries for the first three cupin domains from the database
sample_entries = []
for _, row in db.head(3).iterrows():
    cupin_rng = row["Cupin domain range"]
    if cupin_rng == "NaN" or str(cupin_rng) == "nan":
        continue  # skip entries without a cupin domain
    seq    = extract_domain(row["Sequence"], cupin_rng)
    header = "{}|{}|{}".format(
        row["Hydrazine synthase name"],
        row["Organism"],
        cupin_rng,
    )
    sample_entries.append((header, seq))

# Write the file
OUT_PATH = "/tmp/sample_cupin.fasta"
write_fasta(OUT_PATH, sample_entries)

# Verify by printing the file contents
with open(OUT_PATH) as fh:
    print(fh.read())

---

## 8. Running `separate_domains.py`

The full script processes all 24 entries and writes two FASTA files. Run it from the terminal:

```bash
conda activate comp_analysis

# HMMER must be loaded for entries that need domain scanning
module load HMMER/3.4-gompi-2023a

cd /scratch/p318738/SSA/AAR-COMP-012/AAR-COMP-012-Step2/
python separate_domains.py
```

Expected output files:
- `cupin_domains.fasta` — 17 cupin domain sequences
- `metrs_domains.fasta` — 17 MetRS domain sequences

> **Why 17, not 24?** The database has 24 entries total. Seven are cupin-only, seven are MetRS-only, and ten are di-domain (both). So the cupin FASTA contains 10 (di-domain) + 7 (cupin-only) = 17 sequences. Same logic for MetRS.

> **Key concept:** When running HMMER on the HPC, you must load the HMMER module first with `module load`. The `module` system prevents software version conflicts by keeping each tool in its own environment.

---

## Summary

In this notebook you:
- Learned what protein domains are and why HS has two (cupin and MetRS)
- Understood the FASTA sequence format and how to read/write it in Python
- Traced the logic of `separate_domains.py`: annotated ranges → extract; no annotation → HMMER
- Parsed the two FASTA output files and verified their contents (17 sequences each)

**Next:** Notebook 3 will submit those FASTA sequences to Boltz2 on the HPC cluster for 3D structure prediction.

---

## Try it yourself

**Exercise 1:** Read `cupin_domains.fasta` and print the name, organism and length of each sequence.

```python
# Hint: SeqIO.parse(path, "fasta"), split the .id by "|"
# your code here
```

**Exercise 2:** From `HS_database.csv`, filter and print all enzymes with "Di-domain" architecture, showing only the Name, Organism, Cupin range and MetRS range columns.

```python
# Hint: db[db["Domain Architecture"] == "Di-domain"][[col1, col2, ...]]
# your code here
```

**Exercise 3:** Write a function that takes a protein sequence and a range string like `"28-101"` and returns the sliced subsequence. Test it on the Spb40 MetRS domain (`121-646`).

```python
# Hint: modify extract_domain() from Cell 9 above
# your code here
```